# R04 — Cross-Study Synthesis and Publication Gate

**RepairScopeX · Q1 evidence-strengthening batch (24 Oct 2026)**

Standalone notebook. Checkpointed and isolated. Never alters E10, E11 or V2-14.

## Meta-analysis and submission gate
Runs immediately on CPU with historical E10 facts; for external replication, upload R02/R03 snapshot ZIPs into the Colab `/content` directory, then **rerun this CPU analysis only**. No model inference. It rejects incomplete paired comparisons and does not present absent external experiments as successes.

In [ ]:
import sys,subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','pandas>=2','scipy>=1.12'],check=True)


In [ ]:
import json,zipfile,os,math,pathlib,collections
from scipy.stats import binomtest
import pandas as pd
OUT=pathlib.Path('/content/RSX_Q1_R04_Meta');OUT.mkdir(exist_ok=True)
# E10 prospective results are historical and not substituted for the external benchmark.
historical={'preserve':24,'finalize':25,'remask':21,'regenerate':20,'router':20}
print('Historical E10 fixed evidence:',historical)

# Read only actual completed R02/R03 records. Input ZIPs optional; later rerun this CPU notebook
# after both GPU studies, but never rerun any model inference.
archives=[]
for parent in ['/content','/content/drive/MyDrive/RepairScopeX/Q1_R02_ALoDLM_External',
               '/content/drive/MyDrive/RepairScopeX/Q1_R03_Independent_0p5B']:
    p=pathlib.Path(parent)
    if p.exists():archives += list(p.glob('RSX_Q1_R0*_Snapshot.zip'))
# Also accept files manually uploaded to /content before running.
archives=list({str(p):p for p in archives}.values())
recs=[]
for path in archives:
    with zipfile.ZipFile(path) as z:
        names=[n for n in z.namelist() if n.startswith('records/') and n.endswith('.json')]
        tmp=[json.loads(z.read(n)) for n in names]
        prot=json.loads(z.read('protocol.json'))
        assert len({r['trial_id'] for r in tmp})==len(tmp)
        assert all(r['protocol_hash']==prot['protocol_hash'] for r in tmp)
        for r in tmp:r['archive']=str(path.name)
        recs.extend(tmp)
print('Verified external trial records',len(recs))
if not recs:
    print('No external snapshots found. Historical-only report produced; leave external comparisons pending.')
else:
    df=pd.DataFrame(recs)
    df['correct']=df['grade'].map(lambda g:g['label']=='correct')
    df['ambiguous']=df['grade'].map(lambda g:g['label']=='ambiguous')
    df.to_csv(OUT/'external_trial_ledger.csv',index=False)
    rows=[]
    for (archive,ds),sub in df.groupby(['archive','dataset']):
        pivot=sub.pivot_table(index='case_id',columns='phase',values='correct',aggfunc='first')
        if not {'source','finalize','regenerate'}.issubset(pivot.columns):continue
        pivot=pivot.dropna(subset=['source','finalize','regenerate']); n=len(pivot)
        if not n:continue
        for action in ['finalize','regenerate']:
            win=int((~pivot['source'].astype(bool)&pivot[action].astype(bool)).sum())
            loss=int((pivot['source'].astype(bool)&~pivot[action].astype(bool)).sum())
            rows.append({'archive':archive,'dataset':ds,'n_matched':n,'intervention':action,
                         'source_accuracy':float(pivot['source'].mean()),'intervention_accuracy':float(pivot[action].mean()),
                         'recovery':win,'damage':loss,'p_exact_two_sided':float(binomtest(min(win,loss),win+loss).pvalue) if win+loss else 1.0})
    print(pd.DataFrame(rows).to_string(index=False) if rows else 'Not enough fully matched triples')
    pd.DataFrame(rows).to_csv(OUT/'matched_external_comparisons.csv',index=False)

# Gate for methods-style claims: impossible to pass if no actual independent external cases.
alltrials=len(recs)
publication_gate={'has_external_data':alltrials>0,'historical_frozen_router_negative':True,
    'different_architecture_complete':any('R03' in x.get('archive','') for x in recs),
    'independent_human_scoring_pending':True,
    'note':'Do not claim improved router or generalization without matched complete model-level evidence.'}
(OUT/'publication_gate.json').write_text(json.dumps(publication_gate,indent=2))
with zipfile.ZipFile('/content/RSX_Q1_R04_Meta_Results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir():z.write(f,f.name)
